In [1]:
pip install lime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 13.6 MB/s  0:00:006.6 MB/s eta 0:00:01
Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=ab5f45953fa254c63dfc72facf6dac940a4fa1c9debef9ad58f1bc6a93aa3767
  Stored in directory: /Users/dhruviramaiya/Library/Caches/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lime]━━━━━━ 3/5 [scikit-image]
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
XAI Multimodal LIME Analysis
Guava Maturity Classification — RGB + Thermal Late Fusion
==========================================================
Mirrors the SHAP XAI pipeline but uses LIME (Local Interpretable
Model-agnostic Explanations) via the `lime` package.

Key design decisions:
  1. LimeTabularExplainer operates on the fused feature vector
     (NUM_RGB_FEATS + NUM_THERMAL_FEATS) — same X_fusion used in SHAP.
  2. fusion_predict returns float32 probability arrays (N, NUM_CLASSES)
     — identical to the SHAP version so the same function is reused.
  3. Per-sample explanations are aggregated into per-class mean |weight|
     matrices so we can produce summary, bar, and modality-contribution
     plots that are visually analogous to the SHAP outputs.
  4. ResNet-50 fc head and DenseNet-121 classifier auto-detect Sequential
     vs plain Linear from checkpoint keys (identical fix as SHAP script).
  5. All plt figures explicitly closed to avoid memory leaks.

Install requirements (if not already present):
    pip install lime torch torchvision matplotlib numpy tqdm Pillow
"""

import os
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# DEVICE
# ──────────────────────────────────────────────
DEVICE = torch.device("cpu")

# ──────────────────────────────────────────────
# CONFIG  — edit paths here only
# ──────────────────────────────────────────────
IMG_SIZE    = 224
NUM_CLASSES = 3
CLASS_NAMES = ["mature", "semi_Mature", "Immature"]

DIGITAL_TEST_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/Dataset/Maturity/train test val split for digital/test"
)
THERMAL_TEST_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/Dataset/Maturity/train test val split  thermal/test"
)
RGB_CKPT = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/DeepLearningModels/digital_best_resnet50_guava.pth"
)
THERMAL_CKPT = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/DeepLearningModels/densenet121_thermal_best.pth"
)
OUTPUT_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_EXPLAIN         = 100   # samples to explain (keep low for speed)
LIME_NUM_FEATURES   = 20    # top features per explanation
LIME_NUM_SAMPLES    = 500   # perturbation samples per LIME call (↑ = more accurate, slower)

# ──────────────────────────────────────────────
# TRANSFORM
# ──────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ──────────────────────────────────────────────
# UTILITIES
# ──────────────────────────────────────────────

def _find_linear(module: torch.nn.Module):
    """Return the last nn.Linear found (depth-first) inside module."""
    last = None
    for m in module.modules():
        if isinstance(m, torch.nn.Linear):
            last = m
    return last


def _build_head(in_features: int, num_classes: int, ckpt: dict, key_prefix: str):
    """
    Inspect checkpoint keys to decide whether the saved head was:
      (a) plain Linear                    -> keys: <prefix>.weight / <prefix>.bias
      (b) Sequential(Dropout, Linear)     -> keys: <prefix>.1.weight / <prefix>.1.bias
    Returns a new nn.Module that matches the checkpoint.
    """
    seq_key   = f"{key_prefix}.1.weight"
    plain_key = f"{key_prefix}.weight"

    if seq_key in ckpt:
        return torch.nn.Sequential(
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(in_features, num_classes),
        )
    elif plain_key in ckpt:
        return torch.nn.Linear(in_features, num_classes)
    else:
        candidates = [k for k in ckpt if k.startswith(key_prefix) and k.endswith(".weight")]
        if candidates:
            deepest = max(candidates, key=lambda k: k.count("."))
            w = ckpt[deepest]
            return torch.nn.Linear(in_features, w.shape[0])
        raise KeyError(
            f"Cannot find classifier weights for prefix '{key_prefix}' in checkpoint. "
            f"Available keys (first 20): {list(ckpt.keys())[:20]}"
        )


def load_resnet50(path: str, num_classes: int = NUM_CLASSES) -> torch.nn.Module:
    ckpt  = torch.load(path, map_location=DEVICE)
    model = models.resnet50(pretrained=False)
    model.fc = _build_head(model.fc.in_features, num_classes, ckpt, "fc")
    model.load_state_dict(ckpt, strict=True)
    return model.to(DEVICE).eval()


def load_densenet121(path: str, num_classes: int = NUM_CLASSES) -> torch.nn.Module:
    ckpt  = torch.load(path, map_location=DEVICE)
    model = models.densenet121(pretrained=False)
    model.classifier = _build_head(
        model.classifier.in_features, num_classes, ckpt, "classifier"
    )
    model.load_state_dict(ckpt, strict=True)
    return model.to(DEVICE).eval()


# ──────────────────────────────────────────────
# FEATURE EXTRACTORS
# ──────────────────────────────────────────────

class ResNetFeatureExtractor(torch.nn.Module):
    """ResNet-50 up to and including avgpool -> (N, 2048)."""
    def __init__(self, model):
        super().__init__()
        self.backbone = torch.nn.Sequential(*list(model.children())[:-1])

    def forward(self, x):
        x = self.backbone(x)
        return torch.flatten(x, 1)


class DenseNetFeatureExtractor(torch.nn.Module):
    """DenseNet-121 features + relu + adaptive pool -> (N, 1024)."""
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.pool     = torch.nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=True)
        x = self.pool(x)
        return torch.flatten(x, 1)


# ──────────────────────────────────────────────
# LOAD MODELS
# ──────────────────────────────────────────────
print("Loading RGB model  (ResNet-50)...")
rgb_model = load_resnet50(RGB_CKPT)

print("Loading Thermal model (DenseNet-121)...")
thermal_model = load_densenet121(THERMAL_CKPT)

rgb_extractor     = ResNetFeatureExtractor(rgb_model)
thermal_extractor = DenseNetFeatureExtractor(thermal_model)

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
    NUM_RGB_FEATS     = rgb_extractor(_dummy).shape[1]
    NUM_THERMAL_FEATS = thermal_extractor(_dummy).shape[1]
    NUM_TOTAL_FEATS   = NUM_RGB_FEATS + NUM_THERMAL_FEATS

print(f"   RGB feature dim     : {NUM_RGB_FEATS}")
print(f"   Thermal feature dim : {NUM_THERMAL_FEATS}")
print(f"   Total fusion dim    : {NUM_TOTAL_FEATS}")

# ──────────────────────────────────────────────
# CLASSIFIER WEIGHTS  (for fusion_predict)
# ──────────────────────────────────────────────

def get_last_linear_weights(module):
    lin = _find_linear(module)
    if lin is None:
        raise RuntimeError("No Linear layer found in module.")
    return lin.weight.detach().cpu(), lin.bias.detach().cpu()

rgb_W, rgb_b         = get_last_linear_weights(rgb_model.fc)
thermal_W, thermal_b = get_last_linear_weights(thermal_model.classifier)

assert rgb_W.shape[0]     == NUM_CLASSES, f"RGB classifier output {rgb_W.shape[0]} != {NUM_CLASSES}"
assert thermal_W.shape[0] == NUM_CLASSES, f"Thermal classifier output {thermal_W.shape[0]} != {NUM_CLASSES}"

# ──────────────────────────────────────────────
# IMAGE LOADER
# ──────────────────────────────────────────────

def load_tensor(path: str) -> torch.Tensor:
    return transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)


# ──────────────────────────────────────────────
# EXTRACT FEATURES
# ──────────────────────────────────────────────
print("\nExtracting features from test set...")

rgb_feats_list, thermal_feats_list, label_list = [], [], []

for cls_idx, cls in enumerate(CLASS_NAMES):
    rgb_dir     = os.path.join(DIGITAL_TEST_DIR, cls)
    thermal_dir = os.path.join(THERMAL_TEST_DIR, cls)

    if not os.path.isdir(rgb_dir):
        print(f"   WARNING: Missing RGB dir: {rgb_dir}")
        continue
    if not os.path.isdir(thermal_dir):
        print(f"   WARNING: Missing thermal dir: {thermal_dir}")
        continue

    common = sorted(
        set(os.listdir(rgb_dir)).intersection(os.listdir(thermal_dir))
    )
    if not common:
        print(f"   WARNING: No paired images for class '{cls}'")
        continue

    for fname in tqdm(common, desc=f"   {cls}"):
        try:
            rgb_t = load_tensor(os.path.join(rgb_dir,     fname))
            thm_t = load_tensor(os.path.join(thermal_dir, fname))
        except Exception as exc:
            print(f"      Skipping {fname}: {exc}")
            continue

        with torch.no_grad():
            rf = rgb_extractor(rgb_t).cpu().numpy()[0]
            tf = thermal_extractor(thm_t).cpu().numpy()[0]

        assert rf.shape[0] == NUM_RGB_FEATS
        assert tf.shape[0] == NUM_THERMAL_FEATS

        rgb_feats_list.append(rf)
        thermal_feats_list.append(tf)
        label_list.append(cls_idx)

if not rgb_feats_list:
    raise RuntimeError("No features extracted — please check dataset paths.")

rgb_feats     = np.array(rgb_feats_list,     dtype=np.float32)
thermal_feats = np.array(thermal_feats_list, dtype=np.float32)
labels        = np.array(label_list,         dtype=np.int32)

X_fusion = np.concatenate([rgb_feats, thermal_feats], axis=1)

assert X_fusion.shape[1] == NUM_TOTAL_FEATS, \
    f"X_fusion width {X_fusion.shape[1]} != {NUM_TOTAL_FEATS}"

print(f"\n   Samples collected  : {len(labels)}")
print(f"   X_fusion shape     : {X_fusion.shape}")

# ──────────────────────────────────────────────
# FUSION PREDICT  ->  (N, NUM_CLASSES)  float32
# Identical to SHAP version — LIME requires same callable signature.
# ──────────────────────────────────────────────

def fusion_predict(X: np.ndarray) -> np.ndarray:
    """
    Late-fusion: average softmax probabilities from both modalities.
    Input  : numpy float32  (N, NUM_RGB_FEATS + NUM_THERMAL_FEATS)
    Output : numpy float32  (N, NUM_CLASSES)
    """
    X = X.astype(np.float32)
    Xr = torch.tensor(X[:, :NUM_RGB_FEATS],  dtype=torch.float32)
    Xt = torch.tensor(X[:, NUM_RGB_FEATS:],  dtype=torch.float32)

    with torch.no_grad():
        rgb_probs     = torch.softmax(Xr @ rgb_W.T     + rgb_b,     dim=1)
        thermal_probs = torch.softmax(Xt @ thermal_W.T + thermal_b, dim=1)
        fused = ((rgb_probs + thermal_probs) / 2.0).numpy()

    return fused.astype(np.float32)   # (N, NUM_CLASSES)


# Sanity check
_test_out = fusion_predict(X_fusion[:3])
assert _test_out.shape == (3, NUM_CLASSES), \
    f"fusion_predict shape mismatch: {_test_out.shape}"
assert _test_out.dtype == np.float32
print(f"\nfusion_predict OK -> output shape {_test_out.shape}")

# ──────────────────────────────────────────────
# FEATURE NAMES
# ──────────────────────────────────────────────
feature_names = (
    [f"RGB_{i}"     for i in range(NUM_RGB_FEATS)]     +
    [f"Thermal_{i}" for i in range(NUM_THERMAL_FEATS)]
)
assert len(feature_names) == X_fusion.shape[1], \
    f"feature_names length {len(feature_names)} != {X_fusion.shape[1]}"

# ──────────────────────────────────────────────
# LIME EXPLAINER SETUP
# ──────────────────────────────────────────────
print("\nInitialising LimeTabularExplainer...")

explainer = LimeTabularExplainer(
    training_data   = X_fusion.astype(np.float32),
    feature_names   = feature_names,
    class_names     = CLASS_NAMES,
    mode            = "classification",
    discretize_continuous = False,   # keep continuous — features are CNN embeddings
    random_state    = 42,
)

# ──────────────────────────────────────────────
# RUN LIME ON ALL EXPLAIN SAMPLES
# Aggregate per-sample results into per-class weight matrices
# lime_weights[c][i, f] = LIME weight of feature f for sample i, class c
# ──────────────────────────────────────────────
n_explain = min(MAX_EXPLAIN, len(X_fusion))
X_explain = X_fusion[:n_explain].astype(np.float32)

print(f"Running LIME (explain={n_explain}, "
      f"num_features={LIME_NUM_FEATURES}, "
      f"num_samples={LIME_NUM_SAMPLES})...")

# Pre-allocate: lime_weights is (NUM_CLASSES, n_explain, NUM_TOTAL_FEATS)
lime_weights = np.zeros((NUM_CLASSES, n_explain, NUM_TOTAL_FEATS), dtype=np.float32)

# Build a feature-name -> index mapping for fast lookup
feat_idx = {name: i for i, name in enumerate(feature_names)}

for sample_i in tqdm(range(n_explain), desc="   LIME explanations"):
    exp = explainer.explain_instance(
        data_row        = X_explain[sample_i],
        predict_fn      = fusion_predict,
        num_features    = LIME_NUM_FEATURES,
        num_samples     = LIME_NUM_SAMPLES,
        labels          = list(range(NUM_CLASSES)),
    )
    for cls_idx in range(NUM_CLASSES):
        for feat_name, weight in exp.as_list(label=cls_idx):
            # LIME may return "feature_name <= value" or "feature_name > value"
            # Extract the raw feature name (always the first token before a space)
            raw_name = feat_name.split(" ")[0]
            if raw_name in feat_idx:
                col = feat_idx[raw_name]
                lime_weights[cls_idx, sample_i, col] = float(weight)

print(f"LIME complete - weight tensor shape: {lime_weights.shape}")

# ──────────────────────────────────────────────
# PLOT HELPER
# ──────────────────────────────────────────────

def save_fig(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close("all")
    print(f"   Saved -> {path}")


# ──────────────────────────────────────────────
# PLOT 1 — Averaged Summary  (analogous to SHAP summary)
# Shows mean |LIME weight| averaged over classes and samples.
# Top-N features sorted by importance.
# ──────────────────────────────────────────────
print("\nPlot 1: Summary (averaged over classes)...")

mean_abs_all      = np.abs(lime_weights).mean(axis=(0, 1))   # (F,)
top_n             = LIME_NUM_FEATURES
top_indices       = np.argsort(mean_abs_all)[::-1][:top_n]
top_names         = [feature_names[i] for i in top_indices]
top_vals          = mean_abs_all[top_indices]

# Build a colour list: blue for RGB features, orange for Thermal
bar_colors = [
    "#4C72B0" if feature_names[i].startswith("RGB") else "#DD8452"
    for i in top_indices
]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    y      = np.arange(top_n),
    width  = top_vals[::-1],        # reverse so highest at top
    color  = bar_colors[::-1],
    edgecolor="white",
    linewidth=0.6,
)
ax.set_yticks(np.arange(top_n))
ax.set_yticklabels(top_names[::-1], fontsize=8)
ax.set_xlabel("Mean |LIME Weight|", fontsize=11)
ax.set_title("LIME Feature Importance — Averaged Over Classes", fontsize=13, pad=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4C72B0", label="RGB (ResNet-50)"),
    Patch(facecolor="#DD8452", label="Thermal (DenseNet-121)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
save_fig(os.path.join(OUTPUT_DIR, "lime_summary_averaged.png"))

# ──────────────────────────────────────────────
# PLOT 2 — Per-class dot / scatter plots
# X-axis: feature value (from X_explain), Y-axis: LIME weight for that class.
# One subplot per top feature, coloured by class.
# ──────────────────────────────────────────────
print("\nPlot 2: Per-class dot plots...")

COLORS_CLASS = ["#2ecc71", "#e67e22", "#e74c3c"]   # one per class

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    sv = lime_weights[cls_idx]   # (n_explain, F)

    # Top features for this class
    mean_abs_cls = np.abs(sv).mean(axis=0)   # (F,)
    top_idx      = np.argsort(mean_abs_cls)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(10, 6))
    y_vals   = np.arange(top_n)
    top_vals_cls  = mean_abs_cls[top_idx][::-1]
    top_names_cls = [feature_names[i] for i in top_idx][::-1]

    bc = [
        "#4C72B0" if feature_names[i].startswith("RGB") else "#DD8452"
        for i in top_idx[::-1]
    ]

    ax.barh(y_vals, top_vals_cls, color=bc, edgecolor="white", linewidth=0.6)
    ax.set_yticks(y_vals)
    ax.set_yticklabels(top_names_cls, fontsize=8)
    ax.set_xlabel("Mean |LIME Weight|", fontsize=11)
    ax.set_title(f"LIME Feature Importance — Class: {cls_name}", fontsize=13, pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
    save_fig(os.path.join(OUTPUT_DIR, f"lime_dot_{cls_name}.png"))

# ──────────────────────────────────────────────
# PLOT 3 — Per-class signed weight distribution
# Box-plot showing distribution of raw signed LIME weights
# for top features, for each class.
# ──────────────────────────────────────────────
print("\nPlot 3: Per-class signed weight distribution (box plots)...")

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    sv = lime_weights[cls_idx]   # (n_explain, F)

    mean_abs_cls = np.abs(sv).mean(axis=0)
    top_idx      = np.argsort(mean_abs_cls)[::-1][:top_n]

    data_to_plot  = [sv[:, i] for i in reversed(top_idx)]
    names_to_plot = [feature_names[i] for i in reversed(top_idx)]
    box_colors    = [
        "#4C72B0" if n.startswith("RGB") else "#DD8452"
        for n in names_to_plot
    ]

    fig, ax = plt.subplots(figsize=(10, 7))
    bp = ax.boxplot(
        data_to_plot,
        vert       = False,
        patch_artist=True,
        notch      = False,
        showfliers = False,
        widths     = 0.6,
    )

    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_yticks(np.arange(1, top_n + 1))
    ax.set_yticklabels(names_to_plot, fontsize=8)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Signed LIME Weight", fontsize=11)
    ax.set_title(f"LIME Signed Weight Distribution — Class: {cls_name}",
                 fontsize=13, pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
    save_fig(os.path.join(OUTPUT_DIR, f"lime_boxplot_{cls_name}.png"))

# ──────────────────────────────────────────────
# PLOT 4 — Modality Contribution
# Same concept as SHAP modality contribution bar chart.
# ──────────────────────────────────────────────
print("\nPlot 4: Modality contribution chart...")

mean_abs_global = np.abs(lime_weights).mean(axis=(0, 1))   # (F,)

rgb_importance     = float(mean_abs_global[:NUM_RGB_FEATS].sum())
thermal_importance = float(mean_abs_global[NUM_RGB_FEATS:].sum())
total              = rgb_importance + thermal_importance

print(f"   RGB Contribution     : {rgb_importance:.4f}  "
      f"({100 * rgb_importance / total:.1f}%)")
print(f"   Thermal Contribution : {thermal_importance:.4f}  "
      f"({100 * thermal_importance / total:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
bar_labels = ["RGB\n(ResNet-50)", "Thermal\n(DenseNet-121)"]
bar_values = [rgb_importance, thermal_importance]
bar_colors_mod = ["#4C72B0", "#DD8452"]

bars = ax.bar(bar_labels, bar_values, color=bar_colors_mod,
              edgecolor="white", linewidth=0.8, width=0.45)

for bar, val in zip(bars, bar_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.01,
        f"{val:.4f}\n({100 * val / total:.1f}%)",
        ha="center", va="bottom", fontsize=10,
    )

ax.set_ylabel("Mean |LIME Weight|", fontsize=11)
ax.set_title("Modality Contribution to Fusion Prediction (LIME)", fontsize=12, pad=10)
ax.set_ylim(0, max(bar_values) * 1.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
save_fig(os.path.join(OUTPUT_DIR, "lime_modality_contribution.png"))

# ──────────────────────────────────────────────
# PLOT 5 — Per-class Heatmap  (bonus, no SHAP equiv)
# Rows = top features, Cols = samples, values = signed LIME weight.
# Useful for spotting patterns across samples.
# ──────────────────────────────────────────────
print("\nPlot 5: Per-class feature heatmaps...")

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    sv = lime_weights[cls_idx]   # (n_explain, F)

    mean_abs_cls = np.abs(sv).mean(axis=0)
    top_idx      = np.argsort(mean_abs_cls)[::-1][:top_n]
    heat_data    = sv[:, top_idx].T                    # (top_n, n_explain)
    heat_labels  = [feature_names[i] for i in top_idx]

    fig, ax = plt.subplots(figsize=(min(14, n_explain * 0.15 + 4), 7))
    im = ax.imshow(heat_data, aspect="auto", cmap="RdBu_r",
                   vmin=-np.abs(heat_data).max(),
                   vmax= np.abs(heat_data).max())
    ax.set_yticks(np.arange(top_n))
    ax.set_yticklabels(heat_labels, fontsize=7)
    ax.set_xlabel("Sample index", fontsize=10)
    ax.set_title(f"LIME Weight Heatmap — Class: {cls_name}", fontsize=13, pad=10)
    plt.colorbar(im, ax=ax, label="Signed LIME Weight", shrink=0.8)
    save_fig(os.path.join(OUTPUT_DIR, f"lime_heatmap_{cls_name}.png"))

# ──────────────────────────────────────────────
# DONE
# ──────────────────────────────────────────────
print(f"\nAll LIME outputs written to:\n   {OUTPUT_DIR}")

Loading RGB model  (ResNet-50)...
Loading Thermal model (DenseNet-121)...
   RGB feature dim     : 2048
   Thermal feature dim : 1024
   Total fusion dim    : 3072

Extracting features from test set...


   Immature: 100%|██████████| 462/462 [00:32<00:00, 14.22it/s]



   Samples collected  : 1287
   X_fusion shape     : (1287, 3072)

fusion_predict OK -> output shape (3, 3)

Initialising LimeTabularExplainer...
Running LIME (explain=100, num_features=20, num_samples=500)...


   LIME explanations: 100%|██████████| 100/100 [00:04<00:00, 23.87it/s]


LIME complete - weight tensor shape: (3, 100, 3072)

Plot 1: Summary (averaged over classes)...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_summary_averaged.png

Plot 2: Per-class dot plots...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_dot_mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_dot_semi_Mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_dot_Immature.png

Plot 3: Per-class signed weight distribution (box plots)...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_boxplot_mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_boxplot_semi_Mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_LIME/outputs/lime_boxplot_Immature.png

Plot 4: Modality contribution chart...
   RGB Contribution     : 0.0479  (40.5